In [ ]:
# allows update of external libraries without need to reload package
%load_ext autoreload
%autoreload 2

# Generate synthetic wtg scada data

## Summary

Demonstrate the generation of synthetic wind turbine scada data w or w/o data corruption for demonstrational purpose.

## Results

### Key results

- Features:
  - Provides: Wind speed [m/s], active power [kW], rotor speed [min^-1], pitch angle [°].
  - Messes: Index and values.

### Details

- Physical relations:
  - Wind speed to power: 
    - According to a simple, piecwise linear power curve.
  - Wind speed to rotor speed:
    - Piecewise linear function such that:
      - Strictly below cut-in: zero.
      - Slightly below cut-in: Linear to interpolate between zero and following proportional relation.
      - Between cut-in and rated: Proportional w/ constant tip speed ratio.
      - Between rated and cut out: Constant at rated rotor speed.
      - Above cut-out: Rotor stopped.
    - Single discontinuity at cut-out.
  - Wind speed to pitch angle:
    - Below rated: Constant 1.
    - Between rated and cut-out: ~arccos((v_rated/v)**3). Constant chosen such that at 25m/s angle is approx 25° (constant w/o special interpretation).
    - Above cut-out: Fully pitched to avoid wind.
- Data corruptions:
  - Curtailments due to noise restrictions or shutdowns.
  - Freeze measurements for a random period of time.
  - Drop entire rows for periods of time.
  - Delete entries for a period of time and replace by NaN for single variables.
  - Add spikes for a period of time for single variables.

## Import and setup

In [ ]:
import numpy as np
import pandas as pd

from phoibe.synthetic_data.turbine import DEFAULT_TIME, Time, DEFAULT_WTG, generate_wtg_scada
from phoibe.synthetic_data.turbine import (
    MessUpPipeline,
    create_default_messup_pipeline,
    create_extended_messup_pipeline,
)
from phoibe.synthetic_data._turbine_noise import GeometricSegments, UniformSegments
from phoibe.synthetic_data._turbine_noise import (
    CurtailmentNight,
    CurtailmentToZero,
    Freeze,
    RowDrop,
    Spike,
    DeleteEntries,
)

In [ ]:
A, k = 10, 2
display(DEFAULT_TIME)
display(DEFAULT_WTG)
TIME = Time(start="2026-02-14T04:00:00", freq="10min", periods=59)

## Demonstration

### Generate clean data w/ a basic column set

In [ ]:
df = generate_wtg_scada(A=A, k=k, time=TIME, wtg_type=DEFAULT_WTG, latent_freq=None)
df.columns

### Mess up data

In [ ]:
pipeline = MessUpPipeline(seed=23)

pipeline.add(CurtailmentNight(limit=3200, column="power"))
pipeline.add(CurtailmentToZero(segments=GeometricSegments(n=5, p=0.1), columns_to_keep=["wind_speed"]))
pipeline.add(Freeze(segments=GeometricSegments(n=1, p=0.3), column="power"))
pipeline.add(Freeze(segments=GeometricSegments(n=3, p=0.2), column="rotor_speed"))
pipeline.add(RowDrop(segments=UniformSegments(n=3, min_len=3, max_len=7)))
pipeline.add(DeleteEntries(segments=UniformSegments(n=5, min_len=1, max_len=1), columns=["pitch_angle"]))
pipeline.add(Spike(segments=UniformSegments(n=5, min_len=1, max_len=1), column="power", magnitude=2000))
pipeline.add(Spike(segments=UniformSegments(n=5, min_len=1, max_len=1), column="wind_speed", magnitude=5))

pipeline.apply(df)

### Use dafault provision

In [ ]:
default_pipeline = create_default_messup_pipeline()
default_pipeline.apply(df=df)

### Generate clean data w/ an extended column set

In [ ]:
df_ext = generate_wtg_scada(A=A, k=k, time=DEFAULT_TIME, wtg_type=DEFAULT_WTG, latent_freq="2min")
display(df_ext.columns)

In [ ]:
for key in ["wind_speed", "power", "rotor_speed", "pitch_angle"]:
    if key == "pitch_angle":
        break
    violations_min = df_ext[key] - df_ext[key + "_min"] < -1e-7
    violations_max = df_ext[key + "_max"] - df_ext[key] < -1e-7
    print(f"{key}-violations:\t", violations_min.sum(), violations_max.sum())

df_ext.loc[df_ext["rotor_speed_max"] - df_ext["rotor_speed"] < -1e-7, ["rotor_speed", "rotor_speed_max"]]

In [ ]:
extended_pipeline = create_extended_messup_pipeline(incidence=0.2, level=0.5)
df_ext_noise = extended_pipeline.apply(df=df_ext)
for key in ["wind_speed", "power", "rotor_speed", "pitch_angle"]:
    violations_min = df_ext_noise[key] - df_ext_noise[key + "_min"] < -1e-7
    violations_max = df_ext_noise[key + "_max"] - df_ext_noise[key] < -1e-7
    print(f"{key}-violations:\t", violations_min.sum(), violations_max.sum(), (violations_min | violations_max).sum())

In [ ]:
key = "wind_speed"
key = "power"
key = "rotor_speed"
key = "pitch_angle"
df_ext_noise[key].plot(c="r", label="noise")
ax = df_ext[key].plot(c="b", label="original")
ax.set_title(key)
ax.legend()

In [ ]:
key = "wind_speed"
# key = "power"
key = "rotor_speed"
# key = "pitch_angle"
for suffix, color in zip(["_max", "_min", ""], ["r", "g", "b"]):
    ax = df_ext[key + suffix].plot(c=color, label=key + suffix, linewidth=0.7)
ax.set_title(key)
ax.legend()